# O Problema

## Contexto

Este notebook investiga a relação entre alucinações produzidas por LLMs na geração de código e violações de regras detectadas pelo SonarQube. O objetivo é entender se determinados tipos de alucinação tendem a se manifestar junto com determinadas regras de qualidade de código e, em que nível essa associação ocorre.

## Dados utilizados

Os dados que serão utilizados foram obtido a partir do trabalho de Liu, em que 624 tarefas foram selecionadas de vários datasets, cada tarefa foi submetida com o memso prompt por 5 LLM's distintas. Isso resultou em 3120 implementações de códigos.

Essa implementação foi avaliada com relação à existência ou não de alucinação e, a partir daí foram classficadas as alucinações conforme a taxonomia do trabalho de Liu. Essas alucinações foram classficadas seguindo uma metodologia citada no artigo.

A partir daí, nosso trabalho detectou alertas do SonarQube dessas implementações e, foram classificados em 60 alertas distintos.

## Questões de Pesquisa

> Questão 1: Olhando para cada implementação de código individualmente: as implementações que têm a alucinação X são mais propensas a também ter a violação da regra Y do que as implementações que não têm a alucinação X?

> Questão 2: Tarefas que manifestam alucinação do tipo X (em alguma de suas implementações) tendem a manifestar a violação da regra Y (em alguma de suas implementações)?

## Objetivo geral 

Identificar, para os 720 pares (alucinação × regra), quais associações são estatisticamente significativas e praticamente relevantes, em ambos os níveis de análise, controlando dependência, confundimento e múltiplas comparações.

## Desafios metodológicos

- Dependẽncia: As implementações da mesma tarefa não são independentes pois vêm do mesmo prompt. Por exemplo, se o prompt é diícil todas podem alucinar, se o prompt é fácil pode ser que nenhuma alucine. O erro padrão assume independência e, se isso for ignorado, o erro padrão fica subestimado e o p-valor fica menor do que deveria. Isso pode causar falsos positivos.

- Confundimento: LLM, dataset e level podem afetar tanto alucinação quanto alerta. Por exemplo, Pode ser que alguma LLM alucine mais que outras, ou que dataset, levels ou project diferentes tenham complexidades de tarefas diferentes. 

## Metodologia

### Fase 1 — Preparação dos dados

1.1 Explodir as listas sonar_rules e hallucination_types para colunas binárias.

1.2  Construir a matriz no nível da implementação (3.120 linhas × 12 alucinações + 60 regras).

1.3  Construir a matriz no nível da tarefa (624 linhas × 12 alucinações + 60 regras, critério any).

### Fase 2 — Nível da implementação (QP1)

2.1 Filtro de viabilidade: manter pares (X, Y) com ≥ N ocorrências de X e de Y (definir N).

2.2 Análise exploratória: tabela 2×2 + Fisher + OR + IC 95% (p-valor não inferencial, pois ignora dependência).

2.3 Análise primária: GLMM logístico
    regra_Y ~ aluc_X + model + dataset + level + (1 | tarefa)
    Usar Firth/penalização em caso de separação; reportar não convergências.

2.4 Moderação: teste de razão de verossimilhança entre o modelo acima e o modelo com aluc_X:model.

2.5 FDR (Benjamini-Hochberg) sobre os p-valores do GLMM (2.3).

### Fase 3 — Nível da tarefa (QP2)

3.1 Filtro de viabilidade análogo ao da Fase 2.

3.2 Exploratória: 2×2 + Fisher + OR + IC 95%.

3.3 Primária: GLM logístico com dataset e level (e, opcionalmente, nº de implementações alucinadas).

3.4 FDR sobre os p-valores de 3.3.

3.5 Sensibilidade: critério majority.

### Fase 4 — Síntese (OE5, OE6)

4.1 Comparar os achados das duas análises (interseção, divergências).

4.2 Gerar heatmap 12 × 60, forest plots e Análise de Correspondência.



## Fase 1: Preparação dos dados

# Dados